# PCEN × CNN 反芻聲紋辨識（PyTorch，僅使用公開資料集）

把碩士論文的 PCEN-CNN 訓練流程（原為 TensorFlow/Keras）改用 PyTorch 重寫。

- 資料：Martinez-Rau et al. (2023) 公開資料集的 `JM_rumination`、`JM_grazing`
- 任務：Eating vs Rumination，1 秒窗、0.5 秒步長
- 切分：5 秒不重疊區塊，分層隨機抽樣 80/10/10；同一區塊的窗口只會出現在同一個集合
- 增強：訓練集注入白噪音（SNR 10±5 dB）
- 評估：10 個 seed，報告平均與標準差


In [ ]:
# ── 0. 環境與參數 ─────────────────────────────
import os, zipfile, urllib.request, random, copy
import numpy as np, pandas as pd, librosa
from tqdm import tqdm
from sklearn.model_selection import train_test_split

SR = 16000
SEG_SAMPS = SR                 # 1 秒窗
STRIDE_SAMPS = SR // 2         # 0.5 秒步長
CHUNK_SEC = 5.0                # 切分單位：5 秒不重疊區塊

N_FFT, HOP_LENGTH, N_MELS, FMIN, FMAX = 1024, 256, 64, 20, 2000
PCEN_KW = dict(gain=0.25, bias=2, power=0.5, time_constant=0.4, eps=1e-6)
SNR_MEAN, SNR_STD = 10, 5

LABEL_MAP = {'b': 0, 'x': 0, 'c': 0, 'grazing': 0, 'r': 1, 'chew_rumination': 1}
CLASS_NAMES = ['Eating', 'Rumination']
NUM_CLASSES = len(CLASS_NAMES)

DATA_ROOT = "dataset/acoustic_dairy_cow_dataset-master/data"
DATASET_URL = ("https://gitlab.com/luciano.mrau/acoustic_dairy_cow_dataset/"
               "-/archive/master/acoustic_dairy_cow_dataset-master.zip")
if not os.path.exists(DATA_ROOT):
    urllib.request.urlretrieve(DATASET_URL, "dataset.zip")
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall("dataset")

PUBLIC_FILES = [
    dict(wav=f"{DATA_ROOT}/JM_rumination.wav", csv=f"{DATA_ROOT}/JM_rumination.csv"),
    dict(wav=f"{DATA_ROOT}/JM_grazing.wav",    csv=f"{DATA_ROOT}/JM_grazing.csv"),
]


## 1. 資料切分（5 秒區塊分層抽樣 80/10/10 → 1 秒滑動窗）

In [ ]:
def load_ann(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.xlsx', '.xls'):
        df = pd.read_excel(path, engine='openpyxl')
    else:
        df = pd.read_csv(path, sep=r'[,\t]+', engine='python')
    df.columns = df.columns.str.strip().str.lower()
    df = df.rename(columns={'finish': 'end', 'label': 'code'})
    df['start'] = pd.to_numeric(df['start'], errors='coerce')
    df['end']   = pd.to_numeric(df['end'],   errors='coerce')
    return df.dropna(subset=['start', 'end'])


def collect_chunks(file_list):
    """5 秒不重疊區塊當切分單位；區塊中心點的標注決定它的類別（只用於分層抽樣）。"""
    chunks, audio, anns = [], [], []
    size = int(CHUNK_SEC * SR)
    for fi, info in enumerate(file_list):
        y, _ = librosa.load(info['wav'], sr=SR)
        ann = load_ann(info['csv'])
        audio.append(y); anns.append(ann)
        n0 = len(chunks)
        for s in range(0, len(y) - size + 1, size):
            ct = (s + size / 2) / SR
            row = ann[(ann['start'] <= ct) & (ann['end'] >= ct)]
            if row.empty:
                continue
            code = str(row.iloc[0]['code']).strip().lower()
            if code in LABEL_MAP:
                chunks.append((fi, s, s + size, LABEL_MAP[code]))
        print(f"  {os.path.basename(info['wav'])}: {len(chunks) - n0} 個區塊")
    return chunks, audio, anns


def split_chunks(chunks, random_state=42):
    """分層隨機抽樣 80/10/10。"""
    labels = [c[3] for c in chunks]
    idx = list(range(len(chunks)))
    idx_tv, idx_te = train_test_split(idx, test_size=0.10, random_state=random_state, stratify=labels)
    idx_tr, idx_va = train_test_split(idx_tv, test_size=1/9, random_state=random_state,
                                      stratify=[labels[i] for i in idx_tv])
    pick = lambda ids: [chunks[i] for i in ids]
    return pick(idx_tr), pick(idx_va), pick(idx_te)


def chunks_to_windows(chunks, audio, anns):
    """區塊內以 0.5 秒步長切 1 秒窗；窗口中心點的標注決定標籤。"""
    segs, labs = [], []
    for fi, s0, e0, _ in chunks:
        y, ann = audio[fi], anns[fi]
        for s in range(s0, e0 - SEG_SAMPS + 1, STRIDE_SAMPS):
            ct = (s + SEG_SAMPS / 2) / SR
            row = ann[(ann['start'] <= ct) & (ann['end'] >= ct)]
            if row.empty:
                continue
            code = str(row.iloc[0]['code']).strip().lower()
            if code in LABEL_MAP:
                segs.append(y[s:s + SEG_SAMPS]); labs.append(LABEL_MAP[code])
    return np.asarray(segs, np.float32), np.asarray(labs, int)


print("處理公開資料集...")
chunks, audio, anns = collect_chunks(PUBLIC_FILES)
ch_tr, ch_va, ch_te = split_chunks(chunks)
X_tr_wav, y_tr = chunks_to_windows(ch_tr, audio, anns)
X_va_wav, y_va = chunks_to_windows(ch_va, audio, anns)
X_te_wav, y_te = chunks_to_windows(ch_te, audio, anns)
del audio

for name, y in [('Train', y_tr), ('Val', y_va), ('Test', y_te)]:
    print(f"{name:5s}: {len(y):5d} 窗  Eating/Rumination = {np.bincount(y, minlength=NUM_CLASSES)}")


處理公開資料集...
  JM_rumination.wav: 243 個區塊
  JM_grazing.wav: 405 個區塊
Train:  3405 窗  Eating/Rumination = [2034 1371]
Val  :   411 窗  Eating/Rumination = [246 165]
Test :   427 窗  Eating/Rumination = [256 171]


## 2. PCEN 特徵 ＋ 訓練集噪音注入 ＋ 標準化

In [ ]:
def add_noise(seg, noise_src, snr_mean, snr_std, rng):
    st = rng.integers(0, len(noise_src) - len(seg))
    noise = noise_src[st:st + len(seg)]
    s_pow, n_pow = np.mean(seg ** 2), np.mean(noise ** 2)
    snr = rng.normal(snr_mean, snr_std)
    return seg + noise * np.sqrt(s_pow / (10 ** (snr / 10)) / n_pow)


def pcen_one(y):
    mel = librosa.feature.melspectrogram(y=y.astype(np.float32), sr=SR, n_fft=N_FFT,
                                         hop_length=HOP_LENGTH, n_mels=N_MELS,
                                         fmin=FMIN, fmax=FMAX)
    return librosa.pcen(mel * (2 ** 31), sr=SR, hop_length=HOP_LENGTH, **PCEN_KW)


noise_source = np.random.default_rng(0).normal(0, 0.05, SR * 60).astype(np.float32)   # 白噪音

def extract(segs, noisy=False, seed=42):
    rng = np.random.default_rng(seed)
    feats = []
    for s in tqdm(segs, leave=False):
        if noisy:
            s = add_noise(s, noise_source, SNR_MEAN, SNR_STD, rng)
        feats.append(pcen_one(s))
    T = max(f.shape[1] for f in feats)
    return np.stack([np.pad(f, ((0, 0), (0, T - f.shape[1]))) for f in feats]).astype(np.float32)


F_tr = extract(X_tr_wav, noisy=True)     # 只有訓練集注入噪音
F_va = extract(X_va_wav)
F_te = extract(X_te_wav)
del X_tr_wav, X_va_wav, X_te_wav

mu, sd = F_tr.mean(), F_tr.std() + 1e-8  # 標準化參數只用訓練集
F_tr, F_va, F_te = [(f - mu) / sd for f in (F_tr, F_va, F_te)]
print("特徵形狀:", F_tr.shape, F_va.shape, F_te.shape)   # 預期 (N, 64, 63)
np.savez("pcen_features.npz", F_tr=F_tr, y_tr=y_tr, F_va=F_va, y_va=y_va,
         F_te=F_te, y_te=y_te, mu=mu, sd=sd)


特徵形狀: (3405, 64, 63) (411, 64, 63) (427, 64, 63)


## 3. PyTorch 模型

`Resizing(64,64) → BN → Conv32 → MaxPool → BN → Conv64 → MaxPool → BN → GAP → Dense128 → Dropout0.4 → Dense2`。
與 Keras 版本對齊的設定：BatchNorm `momentum=0.01, eps=1e-3`、Glorot uniform 初始化、Adam `eps=1e-7`、Dense(128) 的 L2(1e-4) 寫成顯式 loss 項、class weight 用 `sum(w·loss)/N`。


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


class CNNBaseline(nn.Module):
    def __init__(self, n_classes=NUM_CLASSES, size=(64, 64)):
        super().__init__()
        self.size = size
        bn = lambda c: nn.BatchNorm2d(c, momentum=0.01, eps=1e-3)
        self.bn0 = bn(3)
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = bn(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = bn(64)
        self.fc1 = nn.Linear(64, 128)
        self.drop = nn.Dropout(0.4)
        self.fc2 = nn.Linear(128, n_classes)
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):                         # x: (N, 1, 64, 63)
        x = x.expand(-1, 3, -1, -1)               # 3 通道複製
        x = F.interpolate(x, size=self.size, mode="bilinear", align_corners=False)
        x = self.bn0(x)
        x = self.bn1(F.max_pool2d(F.relu(self.conv1(x)), 2))
        x = self.bn2(F.max_pool2d(F.relu(self.conv2(x)), 2))
        x = x.mean(dim=(2, 3))                    # GlobalAveragePooling
        x = self.drop(F.relu(self.fc1(x)))
        return self.fc2(x)                        # logits


n_params = sum(p.numel() for p in CNNBaseline().parameters())
print(f"可訓練參數: {n_params:,}")                # 預期 28,168
assert n_params == 28168


device: cuda
可訓練參數: 28,168


## 4. 訓練與評估（EarlyStopping patience=5、ReduceLROnPlateau factor=0.5 patience=3）

In [ ]:
L2_LAMBDA = 1e-4

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def to_tensor(Fm, y):
    return torch.from_numpy(Fm).unsqueeze(1), torch.from_numpy(y).long()


@torch.no_grad()
def predict(model, X, bs=256):
    model.eval()
    return torch.cat([model(X[i:i+bs].to(device)).argmax(1).cpu()
                      for i in range(0, len(X), bs)]).numpy()


@torch.no_grad()
def val_loss_fn(model, X, y, bs=256):
    # 與 Keras 一致：含 L2 項、不加 class weight
    model.eval()
    tot = sum(F.cross_entropy(model(X[i:i+bs].to(device)), y[i:i+bs].to(device),
                              reduction="sum").item() for i in range(0, len(X), bs))
    return tot / len(X) + L2_LAMBDA * model.fc1.weight.pow(2).sum().item()


def train_one_seed(seed, Xtr, ytr, Xva, yva, epochs=50, bs=64):
    set_seed(seed)
    model = CNNBaseline().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, eps=1e-7)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=3, min_lr=1e-6,
        threshold=1e-4, threshold_mode="abs")
    cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=ytr.numpy())
    cw = torch.tensor(cw, dtype=torch.float32, device=device)
    g = torch.Generator().manual_seed(seed)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=bs, shuffle=True, generator=g)

    best, best_state, wait = float("inf"), None, 0
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            ce = F.cross_entropy(model(xb), yb, reduction="none")
            loss = (ce * cw[yb]).mean() + L2_LAMBDA * model.fc1.weight.pow(2).sum()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        vl = val_loss_fn(model, Xva, yva)
        sched.step(vl)
        if vl < best:
            best, best_state, wait = vl, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= 5:
                break
    model.load_state_dict(best_state)             # restore_best_weights
    return model


## 5. 10 個 seed 重複實驗

In [ ]:
Xtr, ytr = to_tensor(F_tr, y_tr)
Xva, yva = to_tensor(F_va, y_va)
Xte, yte = to_tensor(F_te, y_te)

rows, cms = [], []
for seed in range(10):
    model = train_one_seed(seed, Xtr, ytr, Xva, yva)
    pred = predict(model, Xte)
    rows.append(dict(seed=seed,
                     accuracy=accuracy_score(y_te, pred),
                     macro_f1=f1_score(y_te, pred, average="macro"),
                     weighted_f1=f1_score(y_te, pred, average="weighted")))
    cms.append(confusion_matrix(y_te, pred, labels=[0, 1]))
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in rows[-1].items()})
    if seed == 0:
        torch.save(model.state_dict(), "cnn_pcen_seed0.pt")

df = pd.DataFrame(rows)
df.to_csv("results_per_seed.csv", index=False)
print(df.drop(columns="seed").agg(["mean", "std"]).round(4))


{'seed': 0, 'accuracy': 1.0, 'macro_f1': 1.0, 'weighted_f1': 1.0}
{'seed': 1, 'accuracy': 1.0, 'macro_f1': 1.0, 'weighted_f1': 1.0}
{'seed': 2, 'accuracy': 0.9977, 'macro_f1': 0.9976, 'weighted_f1': 0.9977}
{'seed': 3, 'accuracy': 1.0, 'macro_f1': 1.0, 'weighted_f1': 1.0}
{'seed': 4, 'accuracy': 1.0, 'macro_f1': 1.0, 'weighted_f1': 1.0}
{'seed': 5, 'accuracy': 1.0, 'macro_f1': 1.0, 'weighted_f1': 1.0}
{'seed': 6, 'accuracy': 0.9977, 'macro_f1': 0.9976, 'weighted_f1': 0.9977}
{'seed': 7, 'accuracy': 1.0, 'macro_f1': 1.0, 'weighted_f1': 1.0}
{'seed': 8, 'accuracy': 1.0, 'macro_f1': 1.0, 'weighted_f1': 1.0}
{'seed': 9, 'accuracy': 1.0, 'macro_f1': 1.0, 'weighted_f1': 1.0}
      accuracy  macro_f1  weighted_f1
mean    0.9995    0.9995       0.9995
std     0.0010    0.0010       0.0010


In [ ]:
cm_sum = np.sum(cms, axis=0)
print("混淆矩陣（10 個 seed 加總，列正規化 = recall）")
print(pd.DataFrame(np.round(cm_sum / cm_sum.sum(1, keepdims=True), 3),
                   index=[f"True_{c}" for c in CLASS_NAMES],
                   columns=[f"Pred_{c}" for c in CLASS_NAMES]))
print(classification_report(y_te, pred, target_names=CLASS_NAMES))


混淆矩陣（10 個 seed 加總，列正規化 = recall）
                 Pred_Eating  Pred_Rumination
True_Eating            0.999            0.001
True_Rumination        0.000            1.000
              precision    recall  f1-score   support

      Eating       1.00      1.00      1.00       256
  Rumination       1.00      1.00      1.00       171

    accuracy                           1.00       427
   macro avg       1.00      1.00      1.00       427
weighted avg       1.00      1.00      1.00       427

